# Batch Inference vs Real-Time Inference

Not every prediction needs to happen in under 100 ms. Some jobs — scoring millions of users overnight, pre-computing recommendations — are better run as scheduled batch jobs. Choosing the wrong mode wastes money or degrades user experience.

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain the difference between real-time and batch inference in terms of latency and throughput
2. Measure p50/p95 latency for a real-time serving endpoint
3. Run batch inference with chunking and measure items-per-second throughput
4. Choose the right mode for a given scenario using the decision guide

## 1. The Two Modes

**Real-time inference** (online serving):
- One request arrives → predict immediately → return response
- Latency target: < 100 ms (or < 20 ms for high-frequency use cases)
- Examples: fraud detection at checkout, search ranking, content recommendation at page load

**Batch inference** (offline scoring):
- Load all data → predict on thousands/millions at once → save results
- Throughput target: maximize items/second; latency is irrelevant
- Examples: nightly email recommendation generation, weekly churn scores for CRM

## 2. Setup — Train a Model

In [ ]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Test accuracy: {acc:.2%}")

# Save for the serving endpoint
joblib.dump(clf, "/tmp/iris_batch_rt.joblib")
print("Model saved")

## Part 1 — Real-Time Inference

### Building the FastAPI Endpoint

In [ ]:
%%writefile /tmp/realtime_app.py
import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Real-Time Iris Classifier")

model = joblib.load("/tmp/iris_batch_rt.joblib")
CLASSES = ["setosa", "versicolor", "virginica"]

class IrisFeatures(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
async def predict(features: IrisFeatures):
    X = np.array([[features.sepal_length, features.sepal_width,
                   features.petal_length, features.petal_width]])
    class_id = int(model.predict(X)[0])
    return {"prediction": CLASSES[class_id], "class_id": class_id}

### Measuring p50 and p95 Latency

p50 (median) = half of requests are faster than this.
p95 = 95% of requests are faster than this. The p95 is what your worst-case users experience.

In [ ]:
import sys, importlib
sys.path.insert(0, "/tmp")
import realtime_app as rmod
importlib.reload(rmod)
from fastapi.testclient import TestClient

client = TestClient(rmod.app)

# Warm up (first call includes import overhead)
payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
_ = client.post("/predict", json=payload)

# Measure 500 single-prediction requests
N = 500
latencies_ms = []
for _ in range(N):
    t0 = time.perf_counter()
    client.post("/predict", json=payload)
    latencies_ms.append((time.perf_counter() - t0) * 1000)

latencies_ms.sort()
p50 = latencies_ms[int(N * 0.50)]
p95 = latencies_ms[int(N * 0.95)]
p99 = latencies_ms[int(N * 0.99)]

print(f"Real-time inference latency ({N} requests, 1 sample each):")
print(f"  p50 (median): {p50:.2f} ms")
print(f"  p95:          {p95:.2f} ms")
print(f"  p99:          {p99:.2f} ms")
print(f"  max:          {max(latencies_ms):.2f} ms")
print(f"\nReal-time throughput: ~{1000/p50:.0f} requests/second at p50")

### When Real-Time Is Required

- **Fraud detection at checkout** — must score in < 200 ms or transaction times out
- **Search ranking** — results must appear before the user notices latency (< 50 ms)
- **Content recommendation at page load** — recommendation must return before the browser renders
- **Autonomous vehicle decisions** — must complete in < 10 ms at 100 Hz

## Part 2 — Batch Inference

### Generating a Large Dataset

In [ ]:
# Simulate 1000 rows of unlabeled user data that needs scoring
rng = np.random.default_rng(42)
N_BATCH = 1000

df = pd.DataFrame(
    rng.uniform(low=[4.3, 2.0, 1.0, 0.1], high=[7.9, 4.4, 6.9, 2.5], size=(N_BATCH, 4)),
    columns=["sepal_length", "sepal_width", "petal_length", "petal_width"]
)

INPUT_CSV = "/tmp/batch_input.csv"
df.to_csv(INPUT_CSV, index=False)
print(f"Generated {len(df)} rows → {INPUT_CSV}")
print(df.head(3))

### Run Batch Inference — Full DataFrame at Once

In [ ]:
# Load the CSV
batch_df = pd.read_csv(INPUT_CSV)
X_batch = batch_df.values

# Run all 1000 predictions in a single call
t0 = time.perf_counter()
predictions = clf.predict(X_batch)
probabilities = clf.predict_proba(X_batch)
elapsed = time.perf_counter() - t0

CLASSES = ["setosa", "versicolor", "virginica"]
batch_df["prediction"] = [CLASSES[p] for p in predictions]
batch_df["confidence"] = probabilities.max(axis=1).round(3)

OUTPUT_CSV = "/tmp/batch_output.csv"
batch_df.to_csv(OUTPUT_CSV, index=False)

throughput = len(X_batch) / elapsed
print(f"Batch inference: {len(X_batch)} rows in {elapsed*1000:.1f} ms")
print(f"Throughput: {throughput:,.0f} items/second")
print(f"Output saved: {OUTPUT_CSV}")
print(batch_df[["sepal_length", "prediction", "confidence"]].head())

### Chunked Batch Processing

For very large datasets (millions of rows), you can't load everything into memory. Process in chunks of 256 rows at a time.

In [ ]:
CHUNK_SIZE = 256
OUTPUT_CHUNKED = "/tmp/batch_output_chunked.csv"

first_chunk = True
total_rows = 0
t0 = time.perf_counter()

for chunk in pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE):
    X_chunk = chunk.values
    preds = clf.predict(X_chunk)
    probs = clf.predict_proba(X_chunk)

    chunk["prediction"] = [CLASSES[p] for p in preds]
    chunk["confidence"] = probs.max(axis=1).round(3)

    # Append to output CSV (header only on first chunk)
    chunk.to_csv(OUTPUT_CHUNKED, mode="a", header=first_chunk, index=False)
    first_chunk = False
    total_rows += len(chunk)
    print(f"  Processed chunk: {len(chunk)} rows  (total: {total_rows})")

elapsed_chunked = time.perf_counter() - t0
print(f"\nChunked batch: {total_rows} rows in {elapsed_chunked*1000:.1f} ms")
print(f"Output: {OUTPUT_CHUNKED}")

## Part 3 — Throughput Comparison

In [ ]:
# Simulate real-time: send 1000 requests one by one
N_RT = 1000
rt_latencies = []

for row in X_batch:
    pl = {"sepal_length": float(row[0]), "sepal_width": float(row[1]),
          "petal_length": float(row[2]), "petal_width": float(row[3])}
    t0 = time.perf_counter()
    client.post("/predict", json=pl)
    rt_latencies.append(time.perf_counter() - t0)

rt_total = sum(rt_latencies)
rt_throughput = N_RT / rt_total

# Batch throughput
t0 = time.perf_counter()
clf.predict(X_batch)
batch_elapsed = time.perf_counter() - t0
batch_throughput = N_RT / batch_elapsed

print(f"{'Mode':<20} {'Items/sec':<15} {'Total time (1000 items)'}")
print("-" * 55)
print(f"{'Real-time (API)':<20} {rt_throughput:<15,.0f} {rt_total*1000:.1f} ms")
print(f"{'Batch (direct)':<20} {batch_throughput:<15,.0f} {batch_elapsed*1000:.1f} ms")
print(f"\nBatch is {batch_throughput/rt_throughput:.1f}x faster per item than simulated real-time API")
print("(API overhead: serialization, routing, HTTP parsing)")

## Part 4 — Decision Guide

| Scenario | Latency need | Volume | Choose |
|---|---|---|---|
| Fraud detection at checkout | < 200 ms | 1 at a time | Real-time |
| Search result ranking | < 50 ms | 1 query at a time | Real-time |
| Email recommendations (nightly) | None | 10M users | Batch |
| Churn scores for CRM (weekly) | None | 1M customers | Batch |
| Inventory reorder suggestions (daily) | None | 100K items | Batch |
| Autocomplete as user types | < 20 ms | 1 at a time | Real-time |

**Recovery from batch failures:** If a batch job fails at row 500K out of 1M, save a checkpoint. On restart, skip to the checkpoint row. Never re-process rows you already scored — output files should be append-only.

## 5. Summary

In this notebook you:
- Built a FastAPI real-time endpoint and measured p50/p95/p99 latency across 500 requests
- Ran batch inference on 1000 rows in a single `clf.predict()` call and measured throughput
- Implemented chunked batch processing for datasets too large to fit in memory
- Compared items/second: batch is significantly faster per item because it avoids HTTP overhead

**The key trade-off:** real-time minimizes latency for individual requests; batch maximizes throughput for large volumes. They serve different use cases — most production systems need both.

## Self-Check (answer before scrolling up)

1. **Which mode would you use for generating email recommendations overnight for 10 million users?** Explain why real-time would be impractical here.
2. **Why is batch inference more efficient per sample than single-request serving?** Name at least two sources of overhead that batch avoids.
3. **What happens if a batch job fails at row 500K out of 1 million?** How would you design the job so you don't have to restart from row 1?